# 🗂️ Hash Tables -- Runnable Notebook

Companion to [`README.md`](README.md).

A tiny hash table built from scratch (hashing + chaining), then Python's `dict`/`set`/`Counter`, and the classic uses.

## 1. A hash table from scratch (separate chaining)
Each bucket is a small list of `(key, value)` pairs; collisions just mean "append to this bucket's list".

In [ ]:
class SimpleHashTable:
    def __init__(self, capacity=8):
        self.capacity = capacity
        self.buckets = [[] for _ in range(capacity)]   # chaining: one list per bucket
        self.size = 0

    def _index(self, key):
        return hash(key) % self.capacity                # O(1): compute, don't search

    def put(self, key, value):
        i = self._index(key)
        bucket = self.buckets[i]
        for j, (k, _) in enumerate(bucket):
            if k == key:
                bucket[j] = (key, value)                 # update existing key
                return
        bucket.append((key, value))                      # new key
        self.size += 1
        if self.size / self.capacity > 0.75:             # load factor too high -> resize
            self._resize()

    def get(self, key, default=None):
        bucket = self.buckets[self._index(key)]
        for k, v in bucket:
            if k == key:
                return v
        return default

    def _resize(self):
        old_buckets = self.buckets
        self.capacity *= 2                                # double capacity
        self.buckets = [[] for _ in range(self.capacity)]
        self.size = 0
        for bucket in old_buckets:                        # re-hash every existing key
            for k, v in bucket:
                self.put(k, v)


ht = SimpleHashTable(capacity=4)
ht.put("alice", 30)
ht.put("bob", 25)
ht.put("carol", 40)
print("alice ->", ht.get("alice"))
print("missing key ->", ht.get("dave", "not found"))
assert ht.get("alice") == 30
assert ht.get("dave", "not found") == "not found"

# Force enough inserts to trigger a resize, and confirm everything survives it
for i in range(20):
    ht.put(f"key{i}", i)
assert ht.capacity > 4                                    # it grew
assert ht.get("alice") == 30                                # old data survived the resize
assert ht.get("key15") == 15
print("capacity after growth:", ht.capacity)

## 2. Collisions: same bucket, different keys
Demonstrate two keys landing in the same bucket and both being retrievable.

In [ ]:
tiny = SimpleHashTable(capacity=8)
# For an int key, Python's hash(x) == x, so keys 3 and 11 BOTH land in bucket (x % 8) == 3.
tiny.put(3, "three")
tiny.put(11, "eleven")     # guaranteed collision with 3 -- same bucket, different key
assert tiny._index(3) == tiny._index(11)          # confirm they really do collide
# Both keys are still retrieved correctly -- chaining handles the collision transparently.
assert tiny.get(3) == "three" and tiny.get(11) == "eleven"
bucket_sizes = [len(b) for b in tiny.buckets]
print("bucket sizes:", bucket_sizes, "(the bucket at index", tiny._index(3), "holds both colliding keys)")

## 3. Python's `dict` / `set` / `Counter`

In [ ]:
from collections import defaultdict, Counter

d = {}
d["alice"] = 30
assert d.get("alice") == 30
assert d.get("missing", 0) == 0
assert "alice" in d
del d["alice"]
assert "alice" not in d

freq = defaultdict(int)                 # auto-creates missing keys with a default value
for ch in "abracadabra":
    freq[ch] += 1
print("letter frequency:", dict(freq))
assert freq["a"] == 5

counts = Counter(["a", "b", "a", "c", "a"])
print("Counter:", counts)
assert counts.most_common(1) == [("a", 3)]

seen = set()
seen.add("x")
assert "x" in seen and "y" not in seen

## 4. Classic use -- Two Sum via complement lookup (O(n), not O(n^2))

In [ ]:
def two_sum(nums, target):
    """For each number, check if its complement was already seen -- O(1) average per check."""
    seen = {}                                    # value -> index
    for i, x in enumerate(nums):
        need = target - x
        if need in seen:                          # O(1) average membership test
            return [seen[need], i]
        seen[x] = i
    return []

result = two_sum([2, 7, 11, 15], 9)
print("two_sum([2,7,11,15], 9) ->", result)
assert result == [0, 1]

## 5. Classic use -- Group Anagrams (grouping by a derived key)

In [ ]:
from collections import defaultdict

def group_anagrams(words):
    """Words that are anagrams of each other share the same SORTED-LETTERS key."""
    groups = defaultdict(list)
    for w in words:
        key = "".join(sorted(w))                  # anagrams collapse to the same key
        groups[key].append(w)
    return list(groups.values())

result = group_anagrams(["eat", "tea", "tan", "ate", "nat", "bat"])
print("groups:", result)
# Order of groups/items isn't guaranteed -- check membership instead.
result_sets = [set(g) for g in result]
assert {"eat", "tea", "ate"} in result_sets
assert {"tan", "nat"} in result_sets
assert {"bat"} in result_sets

## ✅ Recap
- A hash table computes an **index** from the key instead of searching for it -- `O(1)` average get/put/delete.
- Collisions are unavoidable (pigeonhole principle) -- handled via **chaining** or **open addressing**.
- Resizing when the load factor climbs keeps operations `O(1)` **amortized**, even though a single resize is `O(n)`.
- Python's `dict`/`set`/`Counter`/`defaultdict` are production-grade hash tables -- reach for them first.
- Classic uses: complement lookup (Two Sum), frequency counting, grouping by a derived key, memoization.

Next: [`17_Sorting_Algorithms`](../17_Sorting_Algorithms/README.md).